In [36]:
import pandas as pd

In [37]:
df_metadata = pd.read_csv("../metadata/2026-01-23_consolidated-metadata/table.zambia_all_data_combined.csv", dtype={"sample_id": str})

In [38]:
df_metadata[df_metadata.duplicated(subset=["sample_id"], keep=False)]

,study,sample_id,province,district,ward,healthfac,lat,long,collection_date,rdt_result,parasitemia,extraction_id,take_for_sequencing,screening_method,has_pcr,location


In [39]:
df_experiments = pd.read_csv("../seqdata/all_standard/summaries/all_data/inventory.csv", dtype={"sample_id": str}).query('status != "control"')

In [40]:
print("Unique sample_ids in metadata:", df_metadata["sample_id"].nunique())
print("Unique sample_ids in experiments:", df_experiments["sample_id"].nunique())

Unique sample_ids in metadata: 9528
Unique sample_ids in experiments: 2685


In [41]:
df_metadata["sample_id_no_leading_zeros"] = df_metadata["sample_id"].str.lstrip("0")

In [42]:
df_experiments["sample_id_no_leading_zeros"] = df_experiments["sample_id"].str.lstrip("0")

In [43]:
df_samples_with_no_metadata = df_experiments[~df_experiments["sample_id"].isin(df_metadata["sample_id"])].query('status != "control"')

In [44]:
len(df_samples_with_no_metadata["sample_id"].unique())

119

In [45]:
df_known_excludes = pd.read_csv("../metadata/2026-01-28_missing-metadata-bb/known-excludes.csv", dtype={"sample_id": str})

In [46]:
df_samples_with_no_metadata = df_samples_with_no_metadata.merge(df_known_excludes[['expt_name', 'barcode']], 
                on=['expt_name', 'barcode'], 
                how='left', 
                indicator=True)
df_samples_with_no_metadata = df_samples_with_no_metadata[df_samples_with_no_metadata['_merge'] == 'left_only'].drop('_merge', axis=1)

In [47]:
df_samples_with_no_metadata_match_no_leading_zeros = df_samples_with_no_metadata[df_samples_with_no_metadata["sample_id_no_leading_zeros"].isin(df_metadata["sample_id_no_leading_zeros"])]

In [48]:
print("Unique sample_ids with no metadata:", df_samples_with_no_metadata["sample_id"].nunique())

Unique sample_ids with no metadata: 36


In [49]:
print("Unque sample_ids with no metadata but with no leading zeros match:", df_samples_with_no_metadata_match_no_leading_zeros["sample_id"].nunique())

Unque sample_ids with no metadata but with no leading zeros match: 0


In [50]:
df_samples_with_no_metadata["expt_name"].value_counts()

expt_name
2024-11-06_SLMM064_MIS2024_Batch7           11
2025-05-08_SLMM074_MIS2024_Batch6            7
2024-10-30_SLMM060_MIS2024Batch2             5
2025-06-12_SLMM076_HRP2-32024_Batch8         3
2025-07-03_SLBM008_HRP22024_Batch12          3
2025-05-08_SLMM075_HRP22024_Batch7           3
2024-10-16_SLMM056_MIS2024_Batch1            2
2025-04-04_SLMM072_HRP22024_Batch5           1
2025-08-06_SLMM082_HRP2_2024_Batch15         1
2025-04-03_SLMM070_HRP2 2024_Batch3          1
2024-10-31_SLMM061_MIS2024_Batch1            1
2026-02-04_SeqLib_SLCC002_HRP224_Batch27     1
2025-07-02_SLMM077_HRP2-32024_Batch10        1
Name: count, dtype: int64

In [51]:
df_samples_with_no_metadata_match_no_leading_zeros

,index,expt_name,barcode,sample_id,sample_type,status,sample_id_no_leading_zeros


In [52]:
df_samples_with_no_metadata_excluding_no_leading_zero = df_samples_with_no_metadata[~df_samples_with_no_metadata["sample_id_no_leading_zeros"].isin(df_metadata["sample_id_no_leading_zeros"])]

In [53]:
df_samples_with_no_metadata_excluding_no_leading_zero.to_csv("../metadata/2026-01-28_missing-metadata-bb/samples-with-no-metadata.csv", index=False)

In [54]:
df_metadata["district"].unique()

<StringArray>
['kapiri mposhi',        'mkushi',         'luano',        'mumbwa',
      'chibombo',       'serenje',      'chitambo',       'milengi',
        'chembe',         'mansa',
 ...
         'pemba',        'gwembe',         'monze',      'mazabuka',
    'chikankata',      'chirundu',      'siavonga',  'mwansabombwe',
       'kabompo',       'sesheke']
Length: 106, dtype: str

In [55]:
df_metadata["rdt_result"].value_counts(dropna=False)

rdt_result
pf_neg     4555
pf_pos     3603
invalid    1370
Name: count, dtype: int64

In [56]:
df_metadata.query('rdt_result == "invalid"')["study"].value_counts()

study
HRP23      1369
MIS2024       1
Name: count, dtype: int64

In [57]:
df_not_sequenced = pd.read_csv("../seqdata/all_standard/summaries/HRP23_MIS2024_taken_for_sequencing/summary.samples_qc.csv", dtype={"sample_id": str})

In [58]:
df_not_sequenced = df_not_sequenced.query('status == "not_sequenced"')

In [59]:
df_not_sequenced = df_not_sequenced.merge(df_metadata[["sample_id", "take_for_sequencing"]], on="sample_id", how="left")

In [60]:
df_not_sequenced = df_not_sequenced.merge(df_metadata[["sample_id", "study", "province", "district", "ward", "healthfac", "rdt_result", "screening_method", "has_pcr", "extraction_id"]], on="sample_id", how="left")

In [61]:
df_not_sequenced["take_for_sequencing"].value_counts(dropna=False)

take_for_sequencing
True    284
Name: count, dtype: int64

In [62]:
df_not_sequenced = df_not_sequenced.query("take_for_sequencing == True")

In [63]:
df_pcr = pd.read_csv("../metadata/2026-01-23_consolidated-metadata/table.pcr_data_combined.csv", dtype={"sample_id": str})

In [64]:
df_not_sequenced = df_not_sequenced.merge(df_pcr[["sample_id", 'genus_cp', 'genus_result',
       'pf_cp', 'pf_result', 'screening_method',
       'pf_convpcr']], on="sample_id", how="left")

In [65]:
df_not_sequenced.to_csv("../metadata/2026-01-28_missing-metadata-bb/samples-not-sequenced.csv", index=False)